## 10. Rigorous Validation of the Operator Factorization

We claimed that the second-order wave equation can be factored into two first-order equations:
$$
\partial_{tt} u - \Delta u = (\partial_t - P)(\partial_t + P)u = 0
$$
where $P = (-\Delta)^{1/2}$. Mathematically, this requires the pseudo-differential composition to satisfy:
$$
P∘P = -\Delta
$$

Let's test this identity in two scenarios:
1. **Constant Coefficients (Homogeneous Medium):** The naive algebraic square root works.
2. **Variable Coefficients (Heterogeneous Medium $c(x)$):** The naive square root **fails**, and the `fractional_power` method reveals the hidden **microlocal corrections** required to save the factorization.

In [ ]:
from psiop import PseudoDifferentialOperator
import sympy as sp
from sympy import symbols, Function, diff, simplify, I, sqrt

x, xi = symbols('x xi', real=True)

### Test A: Constant Coefficients (The Trivial Case)

For a homogeneous medium, $-\Delta$ has the symbol $\xi^2$. 
Its square root is $P = |\xi|$. Let's verify that $P∘P = \xi^2$.

In [ ]:
# 1. Define the Laplacian symbol (in 1D, -Delta -> xi^2)
Lap_sym = xi**2
Lap_op = PseudoDifferentialOperator(Lap_sym, [x], mode='symbol')

# 2. Compute the fractional power P = (-Delta)^{1/2}
P_sym = Lap_op.fractional_power(alpha=1/2, order=2, method='symbolic')
P_op = PseudoDifferentialOperator(P_sym, [x], mode='symbol')

# 3. Compose P with itself: P∘P
composed_sym = P_op.compose_asymptotic(P_op, order=1, mode='kn')

print("Symbol of P = (-\\Delta)^{1/2}:")
sp.pprint(P_sym)

print("\nSymbol of P∘P:")
sp.pprint(sp.nsimplify(composed_sym))

print("\nDoes P∘P exactly equal \\xi^2 ?", sp.nsimplify(composed_sym - Lap_sym) == 0)

### Test B: Variable Coefficients (The Microlocal Magic)

Now, consider a wave propagating in a heterogeneous medium with spatially varying speed $c(x)$. 
The spatial operator is $L = -c(x)^2 \partial_x^2$, which has the symbol $l(x, \xi) = c(x)^2 \xi^2$.

If we try to factor this naively, we would guess the square root symbol is $p_{\text{naive}} = c(x)\xi$. 
Let's see what happens when we compose this naive symbol with itself using the Kohn-Nirenberg rule:
$$
(p∘p)(x, \xi) = p^2 - i \partial_\xi p \partial_x p + \dots
$$

Because $c(x)$ and $\xi$ do not commute, **the naive factorization breaks!**

In [ ]:
# Define spatially varying wave speed c(x)
c = Function('c')(x)

# 1. The true heterogeneous operator L = -c(x)^2 \partial_x^2
L_het_sym = c**2 * xi**2
L_het_op = PseudoDifferentialOperator(L_het_sym, [x], mode='symbol')

# 2. The NAIVE square root guess: p_naive = c(x) * xi
p_naive_sym = c * xi
p_naive_op = PseudoDifferentialOperator(p_naive_sym, [x], mode='symbol')

# 3. Compose the naive symbol with itself
naive_composed = p_naive_op.compose_asymptotic(p_naive_op, order=1, mode='kn')

print("--- NAIVE FACTORIZATION ---")
print("Naive symbol p_naive = c(x)*\\xi")
print("Composed p_naive∘p_naive:")
sp.pprint(simplify(naive_composed))

error_term = simplify(naive_composed - L_het_sym)
print(f"\nError (Spurious term generated by non-commutativity):")
sp.pprint(error_term)
print("^ Notice the -i*c(x)*c'(x)*\\xi term! The naive factorization FAILS.")

### The Solution: Asymptotic Fractional Power

To make the factorization $(\partial_t - P)(\partial_t + P) = \partial_t^2 - L$ strictly valid, the symbol of $P$ **must** include lower-order asymptotic corrections that exactly cancel the spurious error terms generated by the composition.

Let's use our `fractional_power` method to compute the **true** symbol of $P = L^{1/2}$, and verify that its composition perfectly reconstructs $L$.

In [ ]:
# 1. Compute the TRUE fractional power P = L^{1/2} using asymptotic expansion
P_true_sym = L_het_op.fractional_power(alpha=0.5, order=1, method='symbolic')
P_true_op = PseudoDifferentialOperator(P_true_sym, [x], mode='symbol')

# 2. Compose the TRUE symbol with itself
true_composed = P_true_op.compose_asymptotic(P_true_op, order=1, mode='kn')

print("--- RIGOROUS FACTORIZATION (via fractional_power) ---")
print("True symbol P = L^{1/2} (includes microlocal corrections):")
try:
    sp.pprint(simplify(P_true_sym))
except TypeError:
    sp.pprint(P_true_sym)

print("\n--- Checking the Factorization Error ---")
try:
    diff_expr = simplify(true_composed - L_het_sym)
except TypeError:
    diff_expr = true_composed - L_het_sym

# 💡 CRITICAL STEP: group terms by their ACTUAL power of ξ, not by sympy's
# default term ordering. `expr.as_ordered_terms()` sorts by an internal
# monomial heuristic that has nothing to do with powers of ξ, so picking
# `terms[0]` can silently drop other terms that sit at the same ξ-order.
# `.coeff(xi, n)` is the correct tool: it collects and SUMS every term
# whose ξ-power is exactly n, for each n we care about.
diff_expr_expanded = sp.expand(diff_expr)

print("Coefficients of (P ∘ P - L) by power of ξ:")
coeffs = {}
for n in range(3, -4, -1):          # scan a generous window of powers
    c_n = sp.simplify(diff_expr_expanded.coeff(xi, n))
    if c_n != 0:
        coeffs[n] = c_n
        print(f"\n  O(ξ^{n}):")
        sp.pprint(c_n)

# Sanity check: confirm we've accounted for the entire expression.
# (If this isn't 0, there's a term outside the scanned power range.)
reconstructed = sum(coeffs.get(n, 0) * xi**n for n in coeffs)
residual = sp.simplify(diff_expr_expanded - reconstructed)
if residual != 0:
    print("\n⚠️  Unaccounted residual outside scanned ξ-powers:")
    sp.pprint(residual)

coeff2 = coeffs.get(2, 0)
coeff1 = coeffs.get(1, 0)
leading_error = coeffs.get(0, 0)   # the FULL O(ξ⁰) term, correctly summed

print(f"\nAre the O(ξ²) and O(ξ¹) terms exactly zero? "
      f"{coeff2 == 0 and coeff1 == 0}")

print(f"\n✅ Full leading error term, O(ξ⁰):")
sp.pprint(leading_error)

print("\n💡 Mathematical Insight:")
if coeff2 == 0 and coeff1 == 0:
    print("The composition proves that the O(ξ²) and O(ξ¹) terms are EXACTLY ZERO!")
    print("The leading error term above is O(ξ⁰), which is the expected asymptotic")
    print("truncation error for an expansion of order=1. The factorization is")
    print("mathematically exact up to the specified asymptotic order!")
else:
    print("Unexpected: a higher-order (ξ² or ξ¹) term did not cancel - check the")
    print("fractional_power / compose_asymptotic implementation for order=1.")

### Conclusion: The Power of Microlocal Calculus

This notebook demonstrated that factoring the wave equation $\partial_{tt} u = c(x)^2 \partial_{xx} u$ into two first-order equations $\partial_t u = \pm P u$ is not a formal algebraic trick, but a genuine pseudodifferential equivalence - one that holds only once $P$ is corrected by a lower-order microlocal term.

1. **The Naive Approach Fails:** Taking the symbol $p_{\text{naive}} = c(x)\xi$ at face value and composing it with itself produces a spurious $-i\,c(x)c'(x)\,\xi$ term - an $O(\xi^1)$ error arising purely from the non-commutativity of multiplication by $c(x)$ and differentiation in $\xi$.

2. **The Rigorous Approach Succeeds:** The asymptotic Newton-type expansion in `fractional_power` finds the correction $P = c(x)\xi + \frac{i}{2}c'(x)$. Composing this corrected symbol with itself exactly cancels the $O(\xi^1)$ term, making the factorization exact through first order.

3. **The WKB Connection:** What remains is a purely $\xi$-independent residual,
$$
\frac{1}{2}c(x)c''(x) - \frac{1}{4}\big(c'(x)\big)^2,
$$
the expected $O(\xi^0)$ truncation error of an order-1 asymptotic expansion, and precisely the classical WKB remainder. The imaginary correction $\frac{i}{2}c'(x)$ is the symbol-level signature of the transport equation that fixes the geometric-optics amplitude $A(x) \propto c(x)^{-1/2}$.

This shows that `psiop`'s symbolic calculus isn't just confirming a known formula - it mechanically derives, term by term, the exact microlocal corrections that a hand derivation would otherwise require.